# B1 generated-only CPU verification v1

In Colab select **Runtime → Change runtime type → CPU**. Run all cells. Upload only `vision_b1_runtime_code_v1.zip` when prompted. No Drive mount, dental images, weights, installs or network downloads. Existing dependencies must import and pass a CPU torch/torchvision compiled-operator check; otherwise STOP.

Acceptance: exactly77 pass, zero failures/errors/skips, both ResNet tests pass, all pinned source hashes unchanged and isolated imports. The tests use weights=None. Evidence downloads even after a dependency/test failure. Colab success is pending execution.


In [ ]:
ZIP_NAME = 'vision_b1_runtime_code_v1.zip'
ZIP_SHA256 = 'd848e87d49c32f259c20ec1127bc6aab0e22e4a825df34dc4bf86522109324ba'
MANIFEST_SHA256 = '5eea8d5689bfd705689f99fbc41cb64a80074f4b52c26b475ff0a485f8af9c19'
EXPECTED_FILES = {'src/vision/colour_b1_prototype.py': 'd3108ea3bf7f310f1b46960174d4936299ece19d762f4ca1ee9d900572b90856', 'src/vision/colour_policy.py': '2bba567545143687f49b6e3ae63fdd72d1eacc10977d5ec373f00a4b5451a97b', 'src/vision/colour_qa.py': '4ad2b81c715b01e624b527d1971cc303740ddf559461f2d953c831956c1e82e3', 'src/vision/exploratory_bundle.py': 'dda776ca1558ed4da795df7cee47c0fe19d382cf32c37ca89d22f4141974b615', 'src/vision/exploratory_bundle_v4.py': 'c914cc05ad2be4ff08873a5d2270b5836e0f84219a539435beeca09b57ad76cd', 'src/vision/exploratory_pilot.py': 'c5bf6cba173390f914a44c4537b91d4f2a8e150aa0f5e4842c2ca94bb1a4d535', 'src/vision/gamut_mapping_feasibility.py': '41cd1d49fcebbb08d45fa1951755d4141a4611630ed9b081f26e5a9b273f368a', 'src/vision/model_pipeline.py': 'c11ad18912575f3567d0baf0eefb1bfbb5eb00fbfa075880fe84aef1ccd34b3c', 'src/vision/train_pipeline.py': 'e7a979b0875af0e459496bca79ce3c80e594acf2639ea7dc9d8659ec75ffc695', 'src/vision/verify_b1_runtime.py': 'bc5f2a2fe17f4e4321d80680b821d93c1b576c607440e29c6331bfe6de3e52f5', 'tests/test_colour_b1_prototype.py': 'db9fb9307b6453f0252ff9d1aef6c2541ca1ec829930e8f803818af122f63e85'}
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, io, json, os, stat, sys, tempfile, zipfile
UPLOADED = files.upload()
if set(UPLOADED) != {ZIP_NAME}:
    raise RuntimeError("Upload only the matching B1 code-only ZIP; no datasets")
archive_bytes = UPLOADED[ZIP_NAME]
if hashlib.sha256(archive_bytes).hexdigest() != ZIP_SHA256:
    raise RuntimeError("Whole ZIP SHA-256 mismatch; STOP")
WORK = Path(tempfile.mkdtemp(prefix="vision_b1_generated_v1_", dir="/content"))
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    entries = archive.infolist()
    expected = set(EXPECTED_FILES) | {"code_manifest.json"}
    if len(entries) != len(expected) or {e.filename for e in entries} != expected:
        raise RuntimeError("Missing, duplicate or extra archive entry; STOP")
    if sum(e.file_size for e in entries) > 2_000_000 or archive.testzip() is not None:
        raise RuntimeError("Archive size/CRC failure; STOP")
    for entry in entries:
        name = entry.filename
        path = PurePosixPath(name)
        mode = entry.external_attr >> 16
        if path.is_absolute() or ".." in path.parts or "\\" in name or not stat.S_ISREG(mode):
            raise RuntimeError("Unsafe code archive path/type; STOP")
        content = archive.read(entry)
        expected_hash = MANIFEST_SHA256 if name == "code_manifest.json" else EXPECTED_FILES[name]
        if hashlib.sha256(content).hexdigest() != expected_hash:
            raise RuntimeError("File hash mismatch: " + name)
        destination = WORK/name
        destination.parent.mkdir(parents=True, exist_ok=True)
        with destination.open("xb") as stream:
            stream.write(content)
manifest = json.loads((WORK/"code_manifest.json").read_text())
if manifest["files"] != EXPECTED_FILES:
    raise RuntimeError("Manifest file allowlist mismatch; STOP")
print("Fresh code-only extraction:", WORK)
print("SHA-256 verified:", len(EXPECTED_FILES), "code/test files plus manifest")


In [ ]:
import subprocess
EVIDENCE = WORK/"synthetic_evidence"
env = dict(os.environ, CUDA_VISIBLE_DEVICES="", PYTEST_DISABLE_PLUGIN_AUTOLOAD="1",
           PYTHONDONTWRITEBYTECODE="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
env.pop("PYTHONPATH", None)
command = [sys.executable, "-I", "-B", str(WORK/"src/vision/verify_b1_runtime.py"), str(WORK)]
try:
    run = subprocess.run(command, cwd=WORK, env=env, text=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, timeout=600)
    output, process_exit = run.stdout, run.returncode
except subprocess.TimeoutExpired as error:
    output = error.stdout or ""
    if isinstance(output, bytes): output = output.decode("utf-8", errors="replace")
    output += "\nRuntime verification exceeded 600 seconds; STOP, not PASS."
    process_exit = -1
EVIDENCE.mkdir(exist_ok=True)
(EVIDENCE/"pytest.log").write_text(output)
print(output)
if not (EVIDENCE/"verification.json").is_file():
    (EVIDENCE/"verification.json").write_text(json.dumps(
        {"status":"FAIL", "blocker":"Runner interrupted before final evidence", "process_exit":process_exit}, indent=2))
print("Next cell downloads evidence for PASS, FAIL or dependency STOP.")


In [ ]:
result = json.loads((EVIDENCE/"verification.json").read_text())
(EVIDENCE/"code_manifest.json").write_bytes((WORK/"code_manifest.json").read_bytes())
(EVIDENCE/"handoff_binding.json").write_text(json.dumps(
    {"zip_sha256":ZIP_SHA256, "manifest_sha256":MANIFEST_SHA256,
     "runner_process_exit":process_exit, "generated_only":True}, indent=2))
evidence_files = sorted(p for p in EVIDENCE.iterdir() if p.is_file())
evidence_hashes = {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in evidence_files}
(EVIDENCE/"evidence_hashes.json").write_text(json.dumps(evidence_hashes, indent=2))
evidence_zip = WORK/"vision_b1_runtime_evidence_v1.zip"
with zipfile.ZipFile(evidence_zip, "x", compression=zipfile.ZIP_DEFLATED) as target:
    for path in sorted(EVIDENCE.iterdir()):
        if path.is_file(): target.write(path, path.name)
print("Evidence ZIP SHA-256:", hashlib.sha256(evidence_zip.read_bytes()).hexdigest())
files.download(str(evidence_zip))
if process_exit != 0 or result.get("status") != "PASS" or result.get("acceptance_passed") is not True:
    raise RuntimeError("B1 runtime verification NOT PASSED. Retain downloaded evidence; no install or fallback.")
print("77 PASSED, 0 failed/errors/skipped; both B1 ResNet-50 tests executed; code unchanged.")
print("Synthetic software verification only, not dental/clinical performance or training approval.")
